# Week 5 Day 2:  LangChain: Tools, Chains, Memory & Your First Framework Agent
Continuing from yesterday, today im rebuilding my raw python agent but using LangChain instead, and then going further with chaining, memory and structured output.

## Goal:
- understand what langchain actually is and what it automates for me
- map what was built yesterday to langchain's building blocks
- build tools, an agent, add memory, and force structured output
- see where langchain hides stuff that i had full control over yesterday

### What is LangChain?
LangChain is a framework that gives you pre built building blocks for making LLM apps and agents, instead of writing everything by hand like w did on day 1. Things like the tool calling loop, memory, chaining prompts together, all of that i wrote manually yesterday (the while loop, messages list, scratchpad etc), langchain basically wraps all of that into reusable classes so i dont have to rewrite the same boilerplate every time. It supports a bunch of different model providers too, so its not tied to one api.

Basically yesterday we were building ourselves, today im using a framework that already built the engine, and i just plug my tools into it. hopefully, itll be easy


## Task 1: LangChain Setup & Core Concepts

Installing langchain + the openai integration for langchain (since we're using our company's base url which is openai compatible, not anthropic's api, so im skipping langchain-anthropic and using langchain-openai instead, same as day 1 where i used the openai client directly)


In [1]:
%pip install langchain==0.3.30 langchain-openai==0.3.32 langchain-community==0.3.29

  Obtaining dependency information for langchain==0.3.30 from https://files.pythonhosted.org/packages/da/54/c0775c29bf9ae27c8cbf8484ad600edd2e787a0eacf3a374cb6561613de9/langchain-0.3.30-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-openai==0.3.32 from https://files.pythonhosted.org/packages/e6/3d/e22ee65fff79afe7bdfbd67844243eb279b440c882dad9e4262dcc87131f/langchain_openai-0.3.32-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-community==0.3.29 from https://files.pythonhosted.org/packages/2b/3c/107819dbed0be3f7a041245bce861c8dd4883ed08040ed482d278a274f22/langchain_community-0.3.29-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-core<1.0.0,>=0.3.85 from https://files.pythonhosted.org/packages/0c/93/ba19ca54701c6118e68f8785949b6c0eab1df3a5cfa5310508cc86877994/langchain_core-0.3.86-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-text-splitters<1.0.0,>=0.3.9 from https://files.pyth

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: C:\Users\imama\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\imama\AppData\Local\Programs\Python\Python311\python.exe
3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]


### Mapping yesterday's raw stuff to LangChain's stuff

Here's how i understand the mapping between what i built and what langchain gives me:

| Day 1 (raw python) | LangChain equivalent | What it does |
|---|---|---|
| `client = OpenAI()` | `ChatOpenAI()` (the LLM wrapper) | wraps the api calls, so i just call `.invoke()` instead of writing `client.chat.completions.create()` every time |
| the `tools` list (json schema dicts) + if/elif to run them | `@tool` decorator / `Tool` class | langchain auto generates the json schema from my python function + docstring, i dont write the schema by hand anymore |
| my `while` loop that checks `tool_calls`, runs the tool, appends to `messages` | `AgentExecutor` | this loop is literally what i wrote yesterday, langchain just runs it for me under the hood |
| `messages = [...]` list i kept appending to | `ConversationBufferMemory` / `RunnableWithMessageHistory` | manages the conversation history for me across turns |

So basically everything i hand rolled yesterday has a langchain class for it now.


### Imports and Setup

In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os

load_dotenv()

from langchain_core.tools import tool #task2


In [4]:

#task 3
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [18]:
# task 5
from pydantic import BaseModel, Field


In [5]:
llm = ChatOpenAI(
    model = "batch",
    api_key = os.getenv("API_KEY"),
    base_url = os.getenv("BASE_URL")
)

#testing
response = llm.invoke("Say Gojo")
print(response.content)

Gojo


### Building a basic pipeline with LCEL

LCEL stands for LangChain Expression Language, its the `|` (pipe) syntax langchain uses to chain components together. Instead of manually passing the output of one step as input to the next, we just pipe them through

```text 
Prompt
|
LLM
|
parser


In [6]:
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer this in one short sentence: {question}"
)

output_parser = StrOutputParser()

#LCEL chain, prompt goes into llm, llms output goes into the parser
chain = prompt | llm | output_parser

result = chain.invoke({"question": "What is Langchain in one line?"})
print(result)

LangChain is a framework for building applications that integrate and orchestrate large language models with external data and tools.


### What is the pipe | doing under the hood?

Every langchain component (prompt, llm, parser) is a "Runnable", meaning it has a standard `.invoke()` method. The `|` operator is python's overload for that, langchain overrides it so `a | b` creates a RunnableSequence that calls `a.invoke()` first and automatically feeds its output into `b.invoke()`. So its basically syntax sugar for `b.invoke(a.invoke(input))`, chained down the line for however many components you pipe together. Nothing magic is happening data wise, its just a  way to wire steps together without writing the passing around manually.


## Task 2: Define & Register Tools

Recreating my day 1 tools (calculator + weather lookup) using langchain's `@tool` decorator, and adding a new tool that reads from a real json "database" file (`products.json`), which i'm using to look up laptop prices for the memory task later.

### Tool 1:  Calculator

In [7]:

#tool 1: Calculator (reused from day 1, just wrapped with @tool now)
@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Perform arithmetic on two numbers. operation must be one of: add, subtract, multiply, divide."""
    if operation == "add":
        return str(a + b)
    elif operation == "subtract":
        return str(a - b)
    elif operation == "multiply":
        return str(a * b)
    elif operation == "divide":
        if b == 0:
            return "Error: Cannot divide by zero."
        return str(a / b)
    else:
        return "Invalid operation."


### Tool 2: Weather lookup

In [8]:
# Tool 2: Weather lookup (reused from day 1)
weather_data = {
    "Lahore": "35°C, Sunny",
    "Karachi": "32°C, Cloudy",
    "Islamabad": "29°C, Rainy",
    "Faisalabad": "34°C, Sunny"
}

@tool
def weather_lookup(city: str) -> str:
    """Return the current weather for a given city."""
    return weather_data.get(city, "Error: Weather data not available.")


### Tool 3:Reads from a real external data source (local json db called products.json) (new)


In [9]:
import json

with open("products.json", "r") as f:
    products_db = json.load(f)

@tool
def product_price_lookup(product_id: str) -> str:
    """Look up a laptop's details (name, price, battery life, ram) from the product database using its id, e.g. laptop_a, laptop_b, laptop_c."""
    product = products_db.get(product_id)
    if not product:
        return f"Error: no product found with id {product_id}"
    return json.dumps(product)


### Why tool docstrings matter

The docstring under each `@tool` function makes it so langchain reads it and sends it to the model as part of the tool's description in the schema, basically the same "description" field as yesterday, except now langchain auto generates it from my docstring instead of me writing it separately. 

So a vague docstring means a vague description sent to the model, which means the model is more likely to pick the wrong tool or use it wrong. Its basically part of the prompt now, so i gotta write it as clearly as i'd write the schema description before


In [10]:
tools = [calculator, weather_lookup, product_price_lookup]

#checking, this is the schema langchain auto built from my docstring
print(calculator.name)
print(calculator.description)
print(calculator.args)


calculator
Perform arithmetic on two numbers. operation must be one of: add, subtract, multiply, divide.
{'operation': {'title': 'Operation', 'type': 'string'}, 'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


### Testing tools further

## Task 3: Build an Agent with create_tool_calling_agent/ AgentExecutor

Now assembling the actual agent. This replaces the entire `while` loop 


In [11]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. Use them when needed to answer the user's question."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [12]:
result = agent_executor.invoke({
    "input": "Look up the weather in Lahore and Karachi. Which city is warmer? Subtract their temperatures and give me the difference."
})

print("\nFinal Answer:")
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `weather_lookup` with `{'city': 'Lahore'}`


35°C, Sunny
Invoking: `weather_lookup` with `{'city': 'Karachi'}`


32°C, Cloudy
Invoking: `calculator` with `{'operation': 'subtract', 'a': 35, 'b': 32}`


3.0Lahore is warmer than Karachi. The temperature difference is **3 °C** (Lahore 35 °C – Karachi 32 °C).

> Finished chain.

Final Answer:
Lahore is warmer than Karachi. The temperature difference is **3 °C** (Lahore 35 °C – Karachi 32 °C).


### Annotating the trace

Looking at the `verbose=True` output above:
- **Reason**: the model looks at my question and decides it needs the weather for both cities before it can compare or subtract anything
- **Act**: it calls `weather_lookup` for Lahore, then again for Karachi
- **Observe**: it gets back the temperature strings from each call
- **Reason again**: now it has both temps, it figures out it needs to pull the numbers out and call `calculator` with subtract
- **Act**: calls `calculator` with the two temps
- **Observe**: gets the numeric difference back
- **Final answer**: puts it all together into one sentence

### Comparing this trace to day 1's raw log

Its basically the exact same reasoning pattern (reason, act, observe, repeat) i saw in my own printed logs yesterday, the ReAct loop is happening either way. Whats different is how much i can *see*. On day 1 i printed every single step myself: the iteration number, the tool name, the arguments, the observation, because i wrote those print statements. Here, `verbose=True` gives me a similar trace but its langchain's format, not mine, and i dont actually see the raw `messages` list building up turn by turn unless i go dig for it. So the loop itself isnt hidden, but the actual data structure moving through it (the message history, the exact tool_call ids etc) is now handled internally and i just have to trust its doing what i think its doing.


## Task 4: Add Memory

Now giving the agent memory so it can handle a follow up conversation, like asking about one laptop, then comparing it to another, then asking for a recommendation, all without me repeating context each time.


In [13]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# simple in memory store, one history per "session"
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


C:\Users\imama\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Wait, my `agent_prompt` doesnt have a `chat_history` placeholder yet, so i need to add that in so the memory actually gets fed back into the prompt.


In [14]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. Use them when needed to answer the user's question."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


In [15]:
config = {"configurable": {"session_id": "client_chat_1"}}

# turn 1
r1 = agent_with_memory.invoke(
    {"input": "Find the price of laptop_a."},
    config=config
)
print(r1["output"])




> Entering new AgentExecutor chain...

Invoking: `product_price_lookup` with `{'product_id': 'laptop_a'}`


{"name": "AeroBook 14", "price": 899, "battery_life_hrs": 10, "ram_gb": 16}The price of **laptop_a** (AeroBook 14) is **$899**.

> Finished chain.
The price of **laptop_a** (AeroBook 14) is **$899**.


In [16]:
# turn 2, follow up, depends on turn 1
r2 = agent_with_memory.invoke(
    {"input": "Now compare it to laptop_c."},
    config=config
)
print(r2["output"])




> Entering new AgentExecutor chain...

Invoking: `product_price_lookup` with `{'product_id': 'laptop_c'}`


{"name": "ValueBook Lite", "price": 549, "battery_life_hrs": 7, "ram_gb": 8}
Invoking: `product_price_lookup` with `{'product_id': 'laptop_a'}`


{"name": "AeroBook 14", "price": 899, "battery_life_hrs": 10, "ram_gb": 16}**Price Comparison**

| Laptop | Price | Battery Life | RAM |
|--------|-------|--------------|-----|
| **AeroBook 14 (laptop_a)** | **$899** | 10 hours | 16 GB |
| **ValueBook Lite (laptop_c)** | **$549** | 7 hours | 8 GB |

- **Price Difference:** AeroBook 14 is **$350 more expensive** than ValueBook Lite.  
- **Value per Dollar:** ValueBook Lite offers a lower price but also has a shorter battery life and half the RAM.  
- **Budget Consideration:** If cost is the primary factor, the $549 ValueBook Lite provides a solid entry‑level option. If you need longer battery life and more RAM for demanding tasks, the $899 AeroBook 14 may be worth the extra investment.

In [17]:
# turn 3, follow up again, depends on both previous turns
r3 = agent_with_memory.invoke(
    {"input": "Which one should I recommend to a budget conscious client?"},
    config=config
)
print(r3["output"])




> Entering new AgentExecutor chain...
For a **budget‑conscious client**, **laptop_c (ValueBook Lite)** is the better recommendation.

### Why laptop_c makes sense

| Factor | laptop_a (AeroBook 14) | laptop_c (ValueBook Lite) |
|--------|------------------------|---------------------------|
| **Price** | $899 | **$549** ( $350 cheaper ) |
| **Battery life** | 10 hours | 7 hours |
| **RAM** | 16 GB | 8 GB |
| **Target use‑case** | Demanding workloads, longer unplugged sessions | Basic productivity (email, web browsing, office apps) |
| **Overall value** | Higher performance at a premium price | Affordable entry‑level laptop that covers everyday tasks |

#### Recommendation
- **If the client’s primary concern is minimizing cost** while still getting a reliable machine for typical office or home use, the **ValueBook Lite (laptop_c)** delivers the essential features at a much lower price point.
- It will comfortably handle web‑based work, document editing, video streaming, and standard p

It worked, the agent remembered laptop_a and laptop_c from the earlier turns without me repeating the ids in turn 3, that's the memory doing its job, its keeping the full chat history (my messages + its tool calls + its answers) and feeding it back in every time.

## Task 5: Structured Output & Error Handling

First, forcing the final recommendation into a structured format using a pydantic model instead of a free text sentence, so its actually usable in code (like if i wanted to save it to a database or send it to another system).



In [19]:
class ProductRecommendation(BaseModel):
    recommended_product: str = Field(description="the name of the recommended laptop")
    price: float = Field(description="the price of the recommended laptop")
    reason: str = Field(description="short reason why this one was picked")

structured_llm = llm.with_structured_output(ProductRecommendation)

structured_result = structured_llm.invoke(
    "Based on this data: laptop_a (AeroBook 14) costs 899 and laptop_c (ValueBook Lite) costs 549, "
    "which one would you recommend to a budget conscious client and why?"
)

print(structured_result)
print(type(structured_result))

recommended_product='ValueBook Lite' price=549.0 reason='It offers a lower price of $549, making it the better choice for a budget‑conscious client who prioritizes cost savings.'
<class '__main__.ProductRecommendation'>


Now for error handling. Adding a tool that fails sometimes on purpose (simulating a flaky api or a bad file read), and seeing how the agent handles it.


In [20]:
import random

@tool
def flaky_stock_checker(product_id: str) -> str:
    """Check live stock availability for a product id. This connects to an external stock service which can be unreliable."""
    if random.random() < 0.5:
        raise ConnectionError("Stock service timed out, please try again.")
    return f"{product_id} is in stock."

tools_with_flaky = [calculator, weather_lookup, product_price_lookup, flaky_stock_checker]

agent = create_tool_calling_agent(llm, tools_with_flaky, agent_prompt)

# handle_parsing_errors + the executor's built in try/except around tool calls
# is what lets it recover instead of crashing the whole run
agent_executor_safe = AgentExecutor(
    agent=agent,
    tools=tools_with_flaky,
    verbose=True,
    handle_tool_error=True,
    handle_parsing_errors=True,
)


In [21]:
result = agent_executor_safe.invoke({
    "input": "Check if laptop_a is in stock."
})
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `flaky_stock_checker` with `{'product_id': 'laptop_a'}`


laptop_a is in stock.Laptop _a_ is currently in stock. Let me know if you’d like more details about the product or want help with purchasing it!

> Finished chain.
Laptop _a_ is currently in stock. Let me know if you’d like more details about the product or want help with purchasing it!


**How it recovers:** by default, if a tool raises an exception, `AgentExecutor` would just crash the whole run. Setting `handle_tool_error=True` tells it to catch the exception, turn it into a message like "tool raised an error: ...", and feed that back to the model as the observation instead of blowing up. So the model actually sees the failure and can decide to retry, apologize, or try another approach, kind of like the error handling i did manually on day 1 where i wrapped tool execution in checks. `handle_parsing_errors=True` does something similar but for when the model's output itself doesnt parse right.

### What langchain made easier vs day 1, and where the magic leaks

Langchain saved me from writing the while loop, the schema generation, and the message bookkeeping by hand, so building an agent with multiple tools and memory took way less code than day 1. The tool schema auto generation from docstrings and the built in memory classes especially saved time. That said, i did notice some abstraction leakiness, when something goes wrong (like a bad prompt template missing a placeholder, which happened to me above) the error messages come from deep inside langchain's internals and arent always obvious, unlike day 1 where every failure was in my own code and easy to trace. Also `AgentExecutor`'s verbose trace is helpful but its not the exact same as seeing the raw `messages` list, so i have less visibility into the literal api calls happening under the hood compared to when i built it myself.


## Summary / Write up

**Raw Python (Day 1) vs LangChain (Day 2):**

- Day 1 gave me full visibility, i wrote every part of the loop, the schema, and the memory myself, so i understood exactly what was being sent to the api at each step, but it meant a lot of repeated boilerplate for every new feature (tools, memory, structured output all needed manual code).
- Day 2 with langchain gave me the same ReAct loop and same core ideas, just packaged into reusable classes (`ChatOpenAI`, `@tool`, `AgentExecutor`, `RunnableWithMessageHistory`), so adding memory or structured output took a few lines instead of rewriting my loop. The tradeoff is less visibility into the raw messages and api calls, and when something breaks the errors come from inside the framework instead of my own code.
- The annotated trace from Task 3 (reason, act, observe, repeat) is basically identical to day 1's, which makes sense since langchain isnt reinventing the ReAct pattern, its just abstracting the code around it.

Overall, langchain is worth it once things get more complex (more tools, memory, multiple agents), but for something small, doing it raw like day 1 keeps things easier to debug.
